<h1><center>Laboratorio 5: La desperación de Mr. Lepin 🐼</center></h1>

<center><strong>MDS7202: Laboratorio de Programación Científica para Ciencia de Datos</strong></center>

---

### Cuerpo Docente

- Profesores: Pablo Badilla y Diego Cortez
- Auxiliares: Valentina Rojas y Melanie Peña
- Ayudantes: Javiera Arévalo, Tamara Carrasco y Ignacio Reyes


### Equipo: SUPER IMPORTANTE - notebooks sin nombre no serán revisados

- Nombre de alumno 1: Isadora Madrid


---

### Reglas

- **Grupos de 2 personas**
- Cualquier duda fuera del horario de clases al foro. Mensajes al equipo docente serán respondidos por este medio.
- Prohibido copiar.
- Uso de LLM (Copilot, Claude, Antigravity, Cursor, etc.) restringido a consultas, documentación y corrección de errores. 
- **Importante**: **¡Recuerden fijar semillas!** Así podemos reproducir sus resultados.

## Descripción del laboratorio.

### Importamos librerias utiles 😸

In [2]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.base import BaseEstimator, TransformerMixin
from typing import Optional


def plot_dim_reductions(
    pca_proj: np.ndarray,
    tsne_proj: np.ndarray,
    umap_proj: np.ndarray,
    name: Optional[str] = None,
    colors: Optional[np.ndarray] = None,
) -> go.Figure:
    fig = make_subplots(rows=1, cols=3, subplot_titles=("PCA", "t-SNE", "UMAP"))

    for i, (proj, title) in enumerate(zip([pca_proj, tsne_proj, umap_proj], ["PCA", "t-SNE", "UMAP"])):
        temp_fig = px.scatter(
            x=proj[:, 0],
            y=proj[:, 1],
            color=colors.astype(str) if colors is not None else None,
            title=title,
            # showlegend=(i == 0),
        )

        for trace in temp_fig.data:
            trace.showlegend = i == 0
            fig.add_trace(trace, row=1, col=i + 1)

    fig.update_layout(height=400, width=1200, title_text=name)
    return fig

# Segmentación de Clientes en Tienda de Retail 🛍️

<p align="center">
  <img width=300 src="https://s1.eestatic.com/2018/04/14/social/la_jungla_-_social_299733421_73842361_854x640.jpg">
</p>

## 1.1 Cargar Dataset

Mr. Lepin, en una nueva reunión, le cuenta a ud y su equipo que los resultados derivados del análisis exploratorio de datos presentaron una gran utilidad para la empresa y que tiene un gran entusiasmo por continuar trabajando con ustedes.
Es por esto, que Mr. Lepin les pide que cargue y visualicen algunas de las filas que componen el Dataset.
A continuación un extracto de lo parlamentado en la reunión:

    - Usted: Es un gran logro para nuestro equipo que usted haya encontrado excelente el EDA. ¿Qué tiene en mente ahora?
    - Mr. Lepin: Resulta que hace algún tiempo, mientras tomaba un mojito en una reunión de gerentes en Panamá, oí a un *chato* acerca de **LRMFP**, que es un modelo que permite personificar a los clientes a través de la fabricación de distintos atributos que describen a los clientes. Lo encontré es-tu-pendo ñatito. 
    - Usted: Ehh bueno. Investigaremos acerca de este modelo y veremos lo que podemos hacer.

Por ende, su siguiente tarea es calcular **LRMFP** sobre cada cliente y luego hacer un análisis de las características generadas. Para esto, el área de ventas les entrega un nuevo archivo llamado `retail_dataset.pickle`, quien posee los datos del DataFrame original limpios y listos para obtener las características solicitadas por Mr. Lepin.

In [3]:
df_retail = pd.read_pickle(
    "https://github.com/MDS7202/MDS7202/raw/refs/heads/main/recursos/2026-01/labs/lab6/retail_dataset.pickle"
)
df_retail = df_retail.astype(
    {
        "Invoice": str,
        "StockCode": str,
        "Description": str,
        "Customer ID": str,
        "Country": str,
    }
)
df_retail.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


## 1.2 Creación de nuevas Caracteristicas [2 Puntos] 

Como ya se les comentó, Mr. Lepin está interesado en obtener las características **LRMFP**, para esto les señala que estas características se construyen en base a las siguientes definiciones:

- **Length (L)**: Intervalo de tiempo, en días, entre la primera y la última visita del cliente. Mientras más grande sea el valor, más fiel es el cliente.

- **Recency (R)**: Indica hace cuánto tiempo el cliente realizó su última compra. Notar que para este caso, mientras más grande es el valor, menos interés posee el usuario para repetir una compra en uno de los locales.

- **Monetary (M)**: El término “monetario” se refiere a la cantidad media de dinero gastada por cada visita del cliente durante el período de observación y refleja la contribución del cliente a los ingresos de la empresa.

- **Frequency (F)**: Se refiere al número total de visitas del cliente durante el periodo de observación. Cuanto mayor sea la frecuencia, mayor será la fidelidad del cliente. 

- **Periodicity (P)**: Representa si los clientes visitan las tiendas con regularidad.

$$Periodicity(n)=std(IVT_1, ..., IVT_n)$$

&nbsp;&nbsp; &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Donde $IVT$ denota el tiempo entre visitas y n representa el número de valores de tiempo entre visitas de un cliente.
 

$$IVT_i=date\_diff(t_{i+1},t_i)$$

En base a las definiciones señaladas, diseñe una función que permita obtener las características **LRMFP** recibiendo un DataFrame como entrada. Para esto, no estará permitido el uso de iteradores; utilicen todas las herramientas que les ofrece `pandas` para realizar esto.

Una referencia que les puede ser útil es el [documento original](https://www.researchgate.net/publication/315979555_LRFMP_model_for_customer_segmentation_in_the_grocery_retail_industry_a_case_study) en donde se propone este método.

**<u>Formato</u> del Resultado Esperado:**

| Customer ID | Length | Recency | Frequency | Monetary | Periodicity |
|------------:|-------:|--------:|----------:|---------:|------------:|
|   12346.0   |    294 |      67 |        46 |   -64.68 |        37.0 |
|   12347.0   |     37 |       3 |        71 |  1323.32 |         0.0 |
|   12349.0   |    327 |      43 |       107 |  2646.99 |        78.0 |
|   12352.0   |     16 |      11 |        18 |   343.80 |         0.0 |
|   12356.0   |     44 |      16 |        84 |  3562.25 |        12.0 |

**Respuesta:**

In [4]:
def custom_features(dataframe_in: pd.DataFrame) -> pd.DataFrame:
    df = dataframe_in.copy()

    # Asegurar que InvoiceDate sea fecha
    df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

    # Crear monto total por fila
    df["TotalPrice"] = df["Quantity"] * df["Price"]

    # Fecha final del periodo de observación
    end_date = df["InvoiceDate"].max()

    # Agrupar por cliente
    features = df.groupby("Customer ID").agg(
        first_purchase=("InvoiceDate", "min"),
        last_purchase=("InvoiceDate", "max"),
        Frequency=("Invoice", "nunique"),
        Monetary=("TotalPrice", "mean")
    ).reset_index()

    # Length
    features["Length"] = (
        features["last_purchase"] - features["first_purchase"]
    ).dt.days

    # Recency
    features["Recency"] = (
        end_date - features["last_purchase"]
    ).dt.days

    # Periodicity
    visits = (
        df[["Customer ID", "InvoiceDate"]]
        .drop_duplicates()
        .sort_values(["Customer ID", "InvoiceDate"])
    )

    visits["IVT"] = visits.groupby("Customer ID")["InvoiceDate"].diff().dt.days

    periodicity = (
        visits.groupby("Customer ID")["IVT"]
        .std()
        .reset_index()
        .rename(columns={"IVT": "Periodicity"})
    )

    # Unir periodicity
    features = features.merge(periodicity, on="Customer ID", how="left")

    # Si tiene una sola visita, no hay desviación estándar
    features["Periodicity"] = features["Periodicity"].fillna(0)

    # Ordenar columnas como pide el enunciado
    features = features[
        ["Customer ID", "Length", "Recency", "Frequency", "Monetary", "Periodicity"]
    ]

    return features

In [5]:
df_features = custom_features(df_retail)
df_features.head()

,Customer ID,Length,Recency,Frequency,Monetary,Periodicity
0,12346.0,196,164,11,11.298788,36.659999
1,12347.0,37,2,2,18.638310,0.000000
2,12348.0,0,73,1,11.108000,0.000000
3,12349.0,181,42,3,26.187647,101.823376
4,12351.0,0,10,1,14.330000,0.000000


## 1.3 Pipelines 👷

Finalmente *Mr. Lepin* le pregunta si sería posible realizar un pipeline para realizar una segmentación de los clientes con los nuevos datos generados, a lo que usted responde que **sí** y propone la utilización de k-means para la segmentación.

A continuación siga los pasos requeridos para obtener la segmentación de clientes.

### 1.3.1 Estandarizar Caracteristicas [0.5 puntos]

Construya una clase llamada ``MinMax()`` utilizando ``BaseEstimator`` y ``TransformerMixin`` para realizar una transformación de cada una de las columnas de un DataFrame utilizando ``ColumnTransformer()`` más tarde (tome como referencia el siguiente [enlace](https://sklearn-template.readthedocs.io/en/latest/user_guide.html#transformer)).


 Para esto considere que Min-Max escaler queda dada por la ecuación:

$$MinMax = \dfrac{x-min(x)}{max(x) - min(x)}$$

Con esto buscamos que los valores que componen a las columnas se muevan en el rango de valores $[0, 1]$.

**Respuesta:**

In [13]:
class MinMax(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        
        self.min_ = X.min()
        self.max_ = X.max()
        
        return self

    def transform(self, X):
        X = pd.DataFrame(X)
        
        denominator = (self.max_ - self.min_).replace(0, 1)
        X_scaled = (X - self.min_) / denominator
        
        return X_scaled.values

In [14]:
#TEST
minmax = MinMax()

cols = ["Length", "Recency", "Frequency", "Monetary", "Periodicity"]

df_scaled = minmax.fit_transform(df_features[cols])

df_scaled[:5]

array([[0.52546917, 0.43967828, 0.04901961, 0.00103152, 0.14361515],
       [0.09919571, 0.00536193, 0.00490196, 0.00170158, 0.        ],
       [0.        , 0.19571046, 0.        , 0.00101411, 0.        ],
       [0.48525469, 0.11260054, 0.00980392, 0.0023908 , 0.39889197],
       [0.        , 0.02680965, 0.        , 0.00130826, 0.        ]])

### 1.3.2 Pipelines de Proyecciones [0.5 puntos]

Para comparar técnicas de reducción de dimensionalidad, realice **tres pipelines** distintos sobre los datos **LRMFP** usando los siguientes métodos:
- **PCA**
- **t-SNE**
- **UMAP**

Para cada pipeline, siga estos pasos:
1. Obtenga las características **LRMFP** desde el DataFrame `retail_dataset.pickle` utilizando la función ``custom_features`` creada anteriormente, junto a ``FunctionTransformer()``. Considere esto como el primer paso de su pipeline.
2. En segundo lugar, usando ``ColumnTransformer()``, aplique el MinMax scaler creado por usted sobre todas las columnas generadas en el paso anterior.
3. Finalmente, aplique el método de reducción de dimensionalidad correspondiente (PCA, t-SNE o UMAP) para obtener las 2 componentes más relevantes.

A continuación, grafique las proyecciones obtenidas de las tres técnicas en una sola figura comparativa.

**Respuesta:**

In [15]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from umap import UMAP

cols_lrfmp = ["Length", "Recency", "Frequency", "Monetary", "Periodicity"]

preprocess = Pipeline([
    ("features", FunctionTransformer(custom_features, validate=False)),
    ("scaler", ColumnTransformer([
        ("minmax", MinMax(), cols_lrfmp)
    ]))
])

pipeline_pca = Pipeline([
    ("preprocess", preprocess),
    ("pca", PCA(n_components=2, random_state=42))
])

pipeline_tsne = Pipeline([
    ("preprocess", preprocess),
    ("tsne", TSNE(n_components=2, random_state=42, init="random", learning_rate="auto"))
])

pipeline_umap = Pipeline([
    ("preprocess", preprocess),
    ("umap", UMAP(n_components=2, random_state=42))
])

In [16]:
# Utilice este código para ejecutar las pipelines y graficar.

pca_proj = pipeline_pca.fit_transform(df_retail)
tsne_proj = pipeline_tsne.fit_transform(df_retail)
umap_proj = pipeline_umap.fit_transform(df_retail)

fig = plot_dim_reductions(
    pca_proj,
    tsne_proj,
    umap_proj,
    name="Reducción de Dimensionalidad",
    colors=None
)

fig.show()

c:\Users\Acer\OneDrive\Escritorio\MDS7202\.venv\lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


### 1.3.3 Análisis de los Loadings de PCA [0.5 puntos]
Antes de continuar con la etapa de clustering, analice los *loadings* (pesos o coeficientes) de las componentes principales obtenidas con PCA. 

Utilice el siguiente tutorial para visualizarlos: https://plotly.com/python/pca-visualization/

- Calcule y reporte los *loadings* de las dos primeras componentes principales.
- Interprete qué características (**LRMFP**) son más relevantes en cada componente.
- Visualice los *loadings* usando un gráfico de barras para cada componente.



In [17]:
# Código para calcular loadings.
import plotly.express as px

# Columnas usadas en PCA
cols_lrfmp = ["Length", "Recency", "Frequency", "Monetary", "Periodicity"]

# Obtener datos ya transformados antes de PCA
X_scaled = preprocess.fit_transform(df_retail)

# Ajustar PCA solo sobre los datos escalados
pca_model = PCA(n_components=2, random_state=42)
pca_model.fit(X_scaled)

# Loadings
loadings = pd.DataFrame(
    pca_model.components_.T,
    columns=["PC1", "PC2"],
    index=cols_lrfmp
)

loadings

,PC1,PC2
Length,0.862529,0.447241
Recency,-0.464568,0.885251
Frequency,0.044528,0.017288
Monetary,-0.001450,0.002559
Periodicity,0.195541,0.126494


In [18]:
#GRAFICAR
loadings_plot = loadings.reset_index().rename(columns={"index": "Feature"})

fig_pc1 = px.bar(
    loadings_plot,
    x="Feature",
    y="PC1",
    title="Loadings PCA - Componente Principal 1"
)

fig_pc1.show()

fig_pc2 = px.bar(
    loadings_plot,
    x="Feature",
    y="PC2",
    title="Loadings PCA - Componente Principal 2"
)

fig_pc2.show()

In [19]:
print("Variables más relevantes en PC1:")
display(loadings["PC1"].abs().sort_values(ascending=False))

print("Variables más relevantes en PC2:")
display(loadings["PC2"].abs().sort_values(ascending=False))

Variables más relevantes en PC1:


Length         0.862529
Recency        0.464568
Periodicity    0.195541
Frequency      0.044528
Monetary       0.001450
Name: PC1, dtype: float64

Variables más relevantes en PC2:


Recency        0.885251
Length         0.447241
Periodicity    0.126494
Frequency      0.017288
Monetary       0.002559
Name: PC2, dtype: float64

### Preguntas sobre loadings:

- ¿Qué son los loadings de PCA?

> Respuesta: Son los coeficientes que indican cuánto contribuye cada variable original a la construcción de cada componente principal. Representan el peso o influencia de cada característica en la dirección de máxima varianza definida por el PCA.

- ¿Qué información relevante obtiene sobre la estructura de los datos a partir de los *loadings* de PCA?

> Respuesta: A partir de los loadings se observa que, la Componente Principal 1 está dominada principalmente por Length (0.86) y en menor medida por Recency (0.46). La Componente Principal 2 está fuertemente dominada por Recency (0.88), seguida por Length (0.44).
Esto indica que la estructura de los datos está fuertemente explicada por variables relacionadas al tiempo del cliente (antigüedad y actividad reciente).
Además, Frequency, Monetary y Periodicity tienen muy poca contribución, lo que sugiere que no explican gran parte de la variabilidad en los datos en comparación con las variables temporales. Entonces, la variabilidad principal del dataset está determinada por el comportamiento temporal de los clientes más que por su gasto o frecuencia.

- ¿Existe alguna relación interesante entre las direcciones de las variables?

> Respuesta: En PC1, Length tiene un loading positivo alto mientras que Recency es negativo, lo que sugiere una relación inversa entre ambas variables en esa componente. Es decir, clientes con mayor antigüedad tienden a tener menor antigüedad de la última compra (más activos recientemente).
En PC2, Length y Recency son positivas, lo que indica que en esta dirección ambas variables tienden a variar conjuntamente.
Las variables Frequency y Monetary tienen valores cercanos a cero en ambas componentes, lo que indica que no están alineadas con las direcciones principales de variabilidad, aportan información menos relevante.

## 1.4 Clustering

### 1.4.1 Método del Codo [0.5 puntos]

Utilizando la clase creada para escalamiento, aplique el método del codo para visualizar cuál es el número de clusters que mejor se ajustan a los datos. Realice esto utilizando el algoritmo K-means dentro de un pipeline para un $k \in [1,20]$, donde k representa el número de clusters del k-means. Para la realización de esta sección y la próxima (1.4.2), considere los mismos pasos utilizados para el t-SNE, pero **permutando el algoritmo de reducción de dimensionalidad por k-means.**

**Respuesta:**

In [20]:
from sklearn.cluster import KMeans   # noqa: F401
import plotly.express as px

ks = range(1, 21)
inertias = []

for k in ks:
    pipeline_kmeans = Pipeline([
        ("preprocess", preprocess),
        ("kmeans", KMeans(n_clusters=k, random_state=42, n_init=10))
    ])
    
    pipeline_kmeans.fit(df_retail)
    
    inertias.append(
        pipeline_kmeans.named_steps["kmeans"].inertia_
    )

df_elbow = pd.DataFrame({
    "k": list(ks),
    "inertia": inertias
})

fig = px.line(
    df_elbow,
    x="k",
    y="inertia",
    markers=True,
    title="Método del Codo para KMeans"
)

fig.show()

df_elbow

,k,inertia
0,1,921.629798
1,2,382.250705
2,3,218.200805
3,4,158.863631
4,5,134.066521
5,6,112.172869
6,7,97.341794
7,8,83.767340
8,9,76.750641
9,10,70.545555


### Preguntas Método del Codo

- A través del gráfico obtenido, comente y justifique qué valor de k escogería para realizar el k-means.

> Respuesta: Al observar el gráfico, se nota que la inercia disminuye muy rápido entre k=1 y k=3, y luego la curva empieza a “aplanarse”. Esto significa que, a partir de ese punto, agregar más clusters ya no reduce tanto la inercia como al inicio. Entre k=3 y k=4 todavía hay una mejora, pero después de k=4 las reducciones son cada vez más pequeñas. Por eso, elegiría k=4, ya que representa un buen equilibrio entre reducir la inercia y no usar más clusters de los necesarios.

- Le fue útil el método del codo para encontrar el número de clusters?

> Respuesta: Sí, ya que permitió identificar un rango razonable de valores para k, específicamente entre 3 y 4, donde ocurre un cambio en la pendiente de la curva.mSin embargo, este método no nos entrega un punto exacto, sino más bien una aproximación visual, por lo que la elección final de k puede requerir criterios adicionales.

- Si no fue así, ¿qué otros métodos podría haber usado para encontrar un número óptimo de clusters?

> Respuesta: Se podrían utilizar, Avarege Silhouette, que mide qué tan bien separado está cada punto respecto a su cluster y a los demás. Gap Statistic, que compara la dispersión observada con la esperada bajo una distribución aleatoria. Métodos jerárquicos (dendrogramas), que permiten visualizar posibles agrupaciones naturales. Computing the number of clusters using R, que permite un análisis exhaustivo, el paquete NbClust proporciona más de 30 índices diferentes para determinar el mejor número de clústeres, combinando métodos de distancia y tipos de algoritmos. 
Estos metodos fueron sacados desde Data Novia.

### 1.4.2 Segmentación de Clientes con K-Means 🎁 [1 punto]

Por último, Mr. Lepin, impaciente de no entender lo que usted intenta explicarle, le solicita que por favor muestre algún resultado "visual y entendible" de los grupos encontrados.

En base a la elección de k realizada en la sección anterior, utilice este valor escogido y entrene un modelo de K-means utilizando el mismo pipeline de scikit-learn utilizado anteriormente.

Una vez ajustado los datos, genere una tabla con los promedios (o medianas) para cada uno de los atributos, agrupando estos por el clúster que pertenecen.

Finalmente, construya un heatmap de las características promedio de cada cluster para visualizar y comparar los perfiles de los grupos.

**Estadísticas de Referencia para K=6:**

Ud. debe calcularlas - Varían de ejecución en ejecución.


|         | Length  | Recency   | Frequency | Monetary | Periodicity |       |
|---------|---------|-----------|----------|-------------|-------|-------|
| Cluster |         |           |          |             |       |       |
|    0    |   258.8 |      45.2 |     76.1 |      1107.7 | 107.6 |   449 |
|    1    |    76.1 |     217.6 |     45.5 |       791.7 |  14.1 |   466 |
|    2    |   368.5 |       4.8 |   2715.0 |    226621.6 |   4.2 |     4 |
|    3    |    85.3 |      45.7 |     65.8 |      1047.0 |  10.5 |   987 |
|    4    |   347.2 |      15.9 |   1658.0 |     35829.3 |   8.0 |    25 |
|    5    |   298.0 |      29.8 |    183.8 |      3639.9 |  32.0 |  1188 |

In [21]:
# Aquí calcule K-Means
from sklearn.cluster import KMeans

k = 4  

pipeline_kmeans = Pipeline([
    ("preprocess", preprocess),
    ("kmeans", KMeans(n_clusters=k, random_state=42, n_init=10))
])

# Entrenar
pipeline_kmeans.fit(df_retail)

# Obtener labels
kmeans_labels = pipeline_kmeans.named_steps["kmeans"].labels_

In [24]:
# Utilice la siguiente función para graficar k-means. kmeans_labels = clusters obtenidos por k-means.
plot_dim_reductions(pca_proj, tsne_proj, umap_proj, name="KMeans K=4", colors=kmeans_labels)

In [26]:
cols_lrfmp = ["Length", "Recency", "Frequency", "Monetary", "Periodicity"]

df_features = custom_features(df_retail)

df_features["Cluster"] = kmeans_labels

cluster_summary = (
    df_features
    .groupby("Cluster")[cols_lrfmp]
    .mean()
)

cluster_summary

,Length,Recency,Frequency,Monetary,Periodicity
Cluster,,,,,
0,318.588235,22.921569,10.159537,34.967766,46.715221
1,17.349765,51.108764,1.699531,33.166186,2.122962
2,181.689311,66.188811,4.269730,27.882352,32.937152
3,17.855422,254.250821,1.500548,54.798705,2.214247


In [27]:
# Aquí grafique el Heatmap
fig = px.imshow(
    cluster_summary,
    text_auto=True,
    aspect="auto",
    title="Heatmap de características promedio por cluster"
)

fig.update_layout(
    xaxis_title="Variables",
    yaxis_title="Cluster"
)

fig.show()

### Preguntas sobre K-Means: 

- ¿Se separaron bien los distintos clusters en cada visualización? 

> Respuesta: En PCA, la separación no es muy clara, ya que es un método lineal que no logra capturar completamente la estructura de los datos. En cambio, en t-SNE y especialmente en UMAP, los clusters se observan más definidos y separados, lo que indica que existen agrupaciones reales en el dataset.

- ¿Es posible observar agrupaciones coherentes?

> Respuesta: Sí. En las proyecciones de t-SNE y UMAP se distinguen grupos relativamente compactos, lo que sugiere que los clusters identificados por K-means representan patrones reales en el comportamiento de los clientes. Además, el heatmap muestra diferencias claras entre los promedios de las variables en cada cluster, lo que refuerza la coherencia de las agrupaciones.

- ¿Quedarían mejor más o menos clusters?

> Respuesta: Considerando el método del codo y las visualizaciones obtenidas, el uso de k=4 parece ser e correcto, ya que permite distinguir grupos con características diferenciadas sin generar una segmentación excesiva. Usar menos clusters podría mezclar perfiles distintos, mientras que usar más clusters podría generar subdivisiones poco interpretables.

- ¿K-Means, dada la forma de las proyecciones, será el mejor método para clusterizar este dataset?¿Habrá algún otro mejor?

> Respuesta: K-Means es una buena primera aproximación, pero tiene limitaciones, ya que asume clusters de forma esférica y puede no capturar estructuras más complejas observadas en t-SNE y UMAP. Dado que las proyecciones muestran formas irregulares, el método de clustering jerárquico podría ser más adecuado para capturar mejor la estructura real de los datos.

Y por último:

- Nombre a cada uno de los clusters según el comportamiento de sus miembros (ej. "C1: Compran poco pero con gran valor...") - Si es necesario, ajuste el número de clusters antes de responder.

> Respuestas: 
* C0 (clientes activos y leales): caracterizados por una alta antigüedad, bajo tiempo desde la última compra y una frecuencia de compra moderada. 
* C1 (clientes ocasionales): con baja antigüedad y muy baja frecuencia de compra, lo que indica una participación limitada. 
* C2 (clientes inactivos): con antigüedad media pero un alto tiempo desde la última compra, lo que sugiere que han dejado de interactuar recientemente.
* C3 (clientes perdidos de alto valor): quienes presentan un alto tiempo desde la última compra pero un gasto relativamente elevado, lo que indica que fueron importantes para la empresa en el pasado pero actualmente no están activos.


## 1.5 Detección de Anomalías con DBSCAN [1 punto]
En esta sección, utilizará el algoritmo DBSCAN para identificar posibles anomalías (outliers) en los clientes del retail.

- Puede aplicar DBSCAN sobre las características originales escaladas (**LRMFP**) o sobre alguna de las proyecciones 2D (PCA, t-SNE o UMAP). Justifique su elección en las preguntas al final de la sección.
- Visualice los resultados usando `plot_dim_reductions`, mostrando los clusters y resaltando los outliers (label = -1) en las tres proyecciones (PCA, t-SNE, UMAP).

In [28]:
from sklearn.cluster import DBSCAN  # noqa: F401

# Obtener datos escalados
X_scaled = preprocess.fit_transform(df_retail)

# DBSCAN
dbscan = DBSCAN(eps=0.3, min_samples=10)
dbscan_labels = dbscan.fit_predict(X_scaled)

dbscan_labels[:10]

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [29]:
# Utilice este código para graficar. dbscan_labels = clusters/outliers obtenidos por DBSCAN.
fig_dbscan = plot_dim_reductions(
    pca_proj,
    tsne_proj,
    umap_proj,
    name="DBSCAN - Detección de Anomalías",
    colors=dbscan_labels
)

fig_dbscan.show()

### Preguntas sobre DBSCAN


1. ¿Por qué decidiste usar los datos originales completos o las proyecciones para aplicar DBSCAN? ¿Por qué no usaste la otra opción?

> Respuesta: Se decidió aplicar DBSCAN sobre las características originales escaladas (LRFMP) porque este algoritmo se basa en distancias entre puntos, por lo que es importante preservar la estructura original de los datos. Las proyecciones como PCA, t-SNE o UMAP reducen la dimensionalidad y pueden distorsionar las distancias reales entre observaciones, lo que podría afectar la detección de anomalías.

2. ¿Cómo elegiste los parámetros de DBSCAN (`eps`, `min_samples`)? ¿Probaste diferentes valores? ¿Cómo afectó esto los resultados?

> Respuesta: Los parámetros se eligieron mediante prueba y error, ajustando principalmente el valor de eps. Se observó que valores muy pequeños generaban muchos puntos clasificados como ruido (-1), mientras que valores muy grandes agrupaban todos los puntos en un solo cluster. Se seleccionó un valor intermedio que permitió identificar un cluster principal junto con algunos outliers. El parámetro min_samples se mantuvo constante en 10, lo que es un valor razonable para evitar que pequeños grupos sean considerados clusters válidos.

3. ¿Tienen sentido los outliers encontrados según el contexto del negocio? ¿Qué interpretación le das a estos clientes? Analiza los datos con pandas si es necesario.

> Respuesta: Sí, los outliers tienen sentido. Estos puntos corresponden a clientes cuyo comportamiento difiere significativamente del resto, ya sea por patrones de compra inusuales, frecuencia extremadamente baja o valores atípicos en gasto o periodicidad. En el gráfico se observa que estos puntos están aislados del cluster principal, lo que respalda su clasificación como anomalías. Estos clientes podrían representar casos especiales, como compradores muy esporádicos, clientes de alto valor con comportamiento irregular o posibles errores en los datos.

# Conclusión
Eso ha sido todo para el lab de hoy, recuerden que el laboratorio tiene un plazo de entrega de una semana. Cualquier duda del laboratorio, no duden en contactarnos por correo, Discord o U-cursos.

![Gracias Totales!](https://i.pinimg.com/originals/65/ae/27/65ae270df87c3c4adcea997e48f60852.gif "bruno")


<br>
<center>
<img src="https://i.kym-cdn.com/photos/images/original/001/194/195/b18.png" width=100 height=50 />
</center>
<br>